In [1]:
import os, shutil, subprocess, sys
sys.path.append('backend/app/')
from hydromt_wflow import WflowSbmModel
from backend.app.services import functions
from backend.app.config import WFLOW_PATH, PROJECT_DES
import numpy as np, geopandas as gpd
np.random.seed(42)
if "bool" not in np.__dict__: np.bool = np.bool_

In [ ]:
# Prepare hydromt
def run_hydromt(mod_path, start, end, time_zone, step, region, resolution, soil_layers, data_lib, uparea, params,
    lulc_function='corine', lulc_mapping_fn='corine_mapping', lai_fn='lai_corine'):
    # Prepare model
    if os.path.exists(mod_path): shutil.rmtree(mod_path)
    os.makedirs(mod_path, exist_ok=True)
    model = WflowSbmModel(
        root=mod_path, config_filename='wflow_sbm.toml', data_libs=data_lib, mode='w'
    )
    start_utc = functions.local_to_utc(start, time_zone).replace(tzinfo=None)
    end_utc = functions.local_to_utc(end, time_zone).replace(tzinfo=None)
    # Setup configurations
    configs = {
        "time.starttime": start_utc, "time.endtime": end_utc, "time.timestepsecs": step,
        # Reference: https://deltares.github.io/Wflow.jl/dev/model_docs/model_settings.html
        'model.type': 'sbm', # model type: [sbm, sbm_gwf]
        'model.cold_start__flag': True,  # Initialize model with cold (cold_start__flag = true) or warm state
        # Unit cell length of input rasters in lat/lon degree (cell_length_in_meter__flag = false) or in meter
        'model.cell_length_in_meter__flag': False, 'model.reservoir__flag': False, #Include reservoir modelling
        'model.water_mass_balance__flag': False, # Include water mass balance error computations
        'model.snow_gravitational_transport__flag': True, # Include gravitational lateral snow transport
        'model.glacier__flag': False, # Include glacier modelling
        'model.soil_infiltration_reduction__flag': False, # Enable reduction factor applied to the soil infiltration capacity
        'model.snow__flag': True, # Include snow modelling
        # Saturated hydraulic conductivity depth profile for SBM soil model
        # optional, one of ("exponential", "exponential_constant", "layered", "layered_exponential"), default is "exponential"
        'model.saturated_hydraulic_conductivity_profile': 'exponential',
        'model.land_routing': 'kinematic_wave', # Routing approach for overland flow: ["kinematic_wave", "local_inertial"]
        'model.river_routing': 'kinematic_wave', # Routing approach for river flow: ["kinematic_wave", "local_inertial"]
        'model.river_kinematic_wave__time_step': 900, 'model.land_kinematic_wave__time_step': 3600,
        'model.kinematic_wave__adaptive_time_step_flag': False, # Enable kinematic wave adaptive (internal) time stepping
        'output.netcdf_grid.path': 'output.nc', 'output.netcdf_grid.compressionlevel': 2,
        # ========== Output variables ==========
        # Source: https://deltares.github.io/Wflow.jl/previews/PR586/model_docs/parameters_routing.html
        # # Overland flow variables
        'output.netcdf_grid.variables.land_surface_water__volume_flow_rate': 'overland_volume_flow', # Overland discharge (average over timestep)	m³ s⁻¹
        'output.netcdf_grid.variables.land_surface_water__depth': 'overland_depth', # Overland depth (average over timestep)	m
        # 'output.netcdf_grid.variables.land_surface_water__volume': 'overland_volume', # Overland volume (average over timestep)	m³
        # 'output.netcdf_grid.variables.land_surface_water__instantaneous_volume_flow_rate': 'overland_discharge_flow', # Discharge overland flow	m³ s⁻¹
        # 'output.netcdf_grid.variables.land_surface_water__instantaneous_depth': 'overland_water_depth_flow', # Water depth overland flow	m
        # # Soil variables
        # 'output.netcdf_grid.variables.land_surface__evapotranspiration_volume_flux': 'soil_evapotranspiration',  # Total actual evapotranspiration	mm
        # 'output.netcdf_grid.variables.land_water~storage~total__depth': 'soil_storage_total',  # Total water storage (excluding floodplains, lakes and reservoirs)	mm
        # 'output.netcdf_grid.variables.soil_water__infiltration_volume_flux': 'soil_infiltration_volume',  # Actual infiltration into the unsaturated zone	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water__transpiration_volume_flux': 'soil_transpiration_volume',  # Transpiration from vegetation	mm Δt⁻¹
        'output.netcdf_grid.variables.soil_surface_water__runoff_volume_flux': 'soil_runoff',  # Total surface runoff from infiltration and saturation excess	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_surface_water__net_runoff_volume_flux': 'soil_net_runoff',  # Net surface runoff (after open water evaporation)	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_layer_water__volume_fraction': 'soil_water_volume_fraction',  # Volumetric water content per soil layer (including residual water content and saturated zone)
        # 'output.netcdf_grid.variables.soil_layer_water__volume_percentage': 'soil_water_volume_percentage',  # Volumetric water content per soil layer (including residual water content and saturated zone)	%
        # 'output.netcdf_grid.variables.soil_water_root-zone__volume_fraction': 'soil_water_rootzone_volume_fraction',  # Volumetric water content in root zone (including residual water content and saturated zone)
        # 'output.netcdf_grid.variables.soil_water_root-zone__volume_percentage': 'soil_water_rootzone_volume_percentage',  # Volumetric water content in root zone (including residual water content and saturated zone)	%
        # 'output.netcdf_grid.variables.soil_water_root-zone__depth': 'soil_water_rootzone_depth',  # Root water storage in unsaturated and saturated zone (excluding residual water content)	mm
        # 'output.netcdf_grid.variables.soil_water_unsat-zone__depth': 'soil_water_unsatzone_depth',  # Amount of water in the unsaturated store	mm
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__capillary_volume_flux': 'soil_water_satzone_capillary_volume_flux',  # Actual capillary rise	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__recharge_volume_flux': 'soil_water_satzone_recharge_volume_flux',  # Downward flux from unsaturated to saturated zone	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__net_recharge_volume_flux': 'soil_water_satzone_net_recharge_volume_flux',  # Net recharge to saturated zone	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_bottom__leakage_volume_flux': 'soil_water_satzone_leakage_volume_flux',  # Actual leakage from saturated store	mm Δt⁻¹
        # 'output.netcdf_grid.variables.soil_water_sat-zone_top__depth': 'soil_water_satzone_depth',  # Pseudo-water table depth (top of the saturated zone)	mm
        
        
        # # Lake variables
        # 'output.netcdf_grid.variables.lake_water__volume': 'lake_volume', # Lake volume (average over timestep), m³
        # 'output.netcdf_grid.variables.lake_water_surface__elevation': 'lake_level', # Lake water level (average over timestep), m
        # 'output.netcdf_grid.variables.lake_water~outgoing__volume_flow_rate': 'lake_outflow', # Outflow of the lake (average over timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.lake_water~incoming__volume_flow_rate': 'lake_inflow', # Inflow into the lake (average over timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.lake_water__evaporation_volume_flux': 'lake_evaporation', # Average actual evaporation over the lake area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.lake_water__precipitation_volume_flux': 'lake_precipitation', # Average precipitation over the lake area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.lake_water__potential_evaporation_volume_flux': 'lake_potential_evaporation', # Average potential evaporation over the lake area	mm Δt⁻¹
        # # Reservoir variables
        # 'output.netcdf_grid.variables.reservoir_water__volume': 'reservoir_volume', # Reservoir volume (average over the timestep)	m³
        # 'output.netcdf_grid.variables.reservoir_water~outgoing__volume_flow_rate': 'reservoir_outflow', # Outflow of the reservoir (average over the timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.reservoir_water~incoming__volume_flow_rate': 'reservoir_inflow', # Inflow into the reservoir (average over the timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.reservoir_water__evaporation_volume_flux': 'reservoir_evaporation', # Average actual evaporation over the reservoir area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.reservoir_water__precipitation_volume_flux': 'reservoir_precipitation', # Average precipitation over the reservoir area	mm Δt⁻¹
        # 'output.netcdf_grid.variables.reservoir_water__potential_evaporation_volume_flux': 'reservoir_potential_evaporation', # Average potential evaporation over the reservoir area	mm Δt⁻¹
        # # River variables (Kinematic wave)
        # 'output.netcdf_grid.variables.river_water__volume_flow_rate': 'river_discharge', # River discharge (average over timestep)	m³ s⁻¹
        # 'output.netcdf_grid.variables.river_water__depth': 'river_depth', # River depth (average over timestep)	m
        # 'output.netcdf_grid.variables.river_water__volume': 'river_volume', # River volume (average over timestep)	m³
        # 'output.netcdf_grid.variables.river_water_inflow~lateral__volume_flow_rate': 'river_lateral_inflow', # Lateral inflow into the river (average over timestep)	m³ s⁻¹
        # # Snow variables
        # 'output.netcdf_grid.variables.snowpack__leq-depth': 'snow_water',  # Liquid-water equivalent of snow pack (SWE)	mm
        # 'output.netcdf_grid.variables.snowpack_meltwater__volume_flux': 'snow_melt',  # Amount of snow melt	mm Δt⁻¹
        # 'output.netcdf_grid.variables.snowpack_water__runoff_volume_flux': 'snow_runoff',  # Runoff from snowpack	mm Δt⁻¹
        # # Glacier variables
        # 'output.netcdf_grid.variables.glacier_ice__melt_volume_flux': 'glacier_melt',  # Melt from the glacier	mm Δt⁻¹
        # # Vegetation variables
        # 'output.netcdf_grid.variables.vegetation_canopy_water__stemflow_volume_flux': 'vegetation_stemflow',  # Stemflow	mm Δt⁻¹
        # 'output.netcdf_grid.variables.vegetation_canopy_water__throughfall_volume_flux': 'vegetation_throughfall',  # Throughfall	mm Δt⁻¹
        
    }
    model.setup_config(configs)
    # Setup basemaps: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_basemaps.html
    model.setup_basemaps(
        region=region, hydrography_fn='my_hydro', res=resolution, upscale_method='ihu' # 'ihu', 'eam', 'dmm'
    )
    # Setup rivers: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_rivers.html
    output_names = {
        'river__length': 'river_length', 'river__width': 'river_width', 'river__slope': 'river_slope',
        'river_bank_water__depth': 'river_bank_depth', # Bankfull depth of river, default is 1.0 m
        'river_water_flow__manning_n_parameter': 'river_manning_n', # Manning's roughness, default is 0.036
        'river_bank_water__elevation': 'river_bank_elevation', 'river_location__mask': 'river_mask'
    }
    model.setup_rivers(
        hydrography_fn='my_hydro', river_geom_fn='river_network', 
        river_upa=uparea, # Minimum upstream area threshold for the river map [km2]
        rivdph_method='powlaw', # 'gvf', 'manning', 'powlaw'
        slope_len=2, #  Length over which the river slope is calculated [km]
        min_rivlen_ratio=0, min_rivdph=1.0, # Minimum river depth [m]
        min_rivwth=30, # Minimum river width [m]
        smooth_len=5000, # Length [m] over which to smooth the output river width and depth
        connectivity=8, river_routing='kinematic_wave', # 'kinematic_wave', 'local_inertial'
        elevtn_map='land_elevation', # Name of the elevation map in the current WflowBaseModel.staticmaps
        output_names=output_names
    )
    model.setup_river_roughness(
        rivman_mapping_fn='river_manning_mapping', # Name of the river manning n map in the current WflowBaseModel.river_maps
        strord_name='meta_streamorder', # Name of the stream order map in the current WflowBaseModel.staticmaps
        output_name='river_manning_n' # Mapping of output variable names.
    )
    # Setup soil maps: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_soilmaps.html
    soil_names = {
        'soil__thickness': 'soil_thickness', 'soil_layer_water__brooks_corey_exponent': 'soil_brooks_corey_c',
        'soil_surface_water__vertical_saturated_hydraulic_conductivity': 'soil_ksat_vertical', 
        'soil_water__residual_volume_fraction': 'soil_theta_r', 'soil_water__saturated_volume_fraction': 'soil_theta_s', 
        'soil_water__vertical_saturated_hydraulic_conductivity_scale_parameter': 'soil_f'
    }
    model.setup_soilmaps(
        soil_fn='soilgrids', ptf_ksatver='brakensiek', # 'brakensiek', 'cosby'
        wflow_thicknesslayers=soil_layers, # Thickness of soil layers [mm] for wflow_sbm soil model
        output_names=soil_names
    )
    model.setup_laimaps_from_lulc_mapping(lulc_fn=lulc_function, lai_mapping_fn=lai_fn)
    # Setup land use maps: https://deltares.github.io/hydromt_wflow/stable/api/_generated/hydromt_wflow.WflowSbmModel.setup_lulcmaps.html
    lulc_variables = [
        'landuse', 'vegetation_kext', 'land_manning_n', 'soil_compacted_fraction', 
        'vegetation_root_depth', 'vegetation_leaf_storage', 'vegetation_wood_storage', 
        'land_water_fraction', 'vegetation_crop_factor', 'vegetation_feddes_alpha_h1', 
        'vegetation_feddes_h1', 'vegetation_feddes_h2', 'vegetation_feddes_h3_high', 
        'vegetation_feddes_h3_low', 'vegetation_feddes_h4'
    ]
    model.setup_lulcmaps(
        lulc_fn=lulc_function, lulc_mapping_fn=lulc_mapping_fn, lulc_vars=lulc_variables
    )
    # Setup forcing
    model.setup_precip_forcing(precip_fn='weather_forcing')
    model.setup_temp_pet_forcing(
        temp_pet_fn='weather_forcing', pet_method='debruin', # 'debruin', 'makkink', 'penman-monteith_rh_simple', 'penman-monteith_tdew'
        press_correction=True, temp_correction=True, wind_correction=True,
        wind_altitude=10, reproj_method='nearest', fillna_method='nearest',
        dem_forcing_fn='dtm', skip_pet=False
    )
    # === Constant parameters ===
    model.setup_constant_pars(
        subsurface_water__horizontal_to_vertical_saturated_hydraulic_conductivity_ratio = params['k_sat_ratio'],
        snowpack__degree_day_coefficient = params['dd_snow'],
        soil_surface_water__infiltration_reduction_parameter = params['inf_red'],
        vegetation_canopy_water__mean_evaporation_to_mean_precipitation_ratio = params['canopy_evap_ratio'],
        compacted_soil_surface_water__infiltration_capacity = params['inf_cap'],
        soil_water_saturated_zone_bottom__max_leakage_volume_flux = params['leak_max'],
        soil_wet_root__sigmoid_function_shape_parameter = params['soil_sigmoid'],
        atmosphere_air__snowfall_temperature_threshold = params['snowfall_t0'],
        atmosphere_air__snowfall_temperature_interval = params['snowfall_dt'],
        snowpack__melting_temperature_threshold = params['melt_t0_snow'],
        snowpack__liquid_water_holding_capacity = params['snow_liq_cap'],
        glacier_ice__degree_day_coefficient = params['dd_glacier'],
        glacier_firn_accumulation__snowpack_dry_snow_leq_depth_fraction = params['firn_dry_frac'],
        glacier_ice__melting_temperature_threshold = params['melt_t0_glacier']
    )
    # === Cold states ===
    model.setup_cold_states()
    # === Write model ===
    model.write(
        grid_filename='static_grid.nc', geoms_folder='staticgeoms', 
        forcing_filename='weather_forcing.nc', states_filename='output_state.nc'
    )

## Prepare data using HydroMT

In [ ]:
test_folder = r"backend\projects\vlnguyen\demo\flows\flow_abc"

lat, lon = 62.45871949688421, 6.355327634388007
start, end, step = "2025-01-01 00:00:00", "2025-02-01 00:00:00", 3600
subbasin = gpd.GeoDataFrame(geometry=gpd.points_from_xy(x=[lon], y=[lat]), crs='EPSG:4326')
subbasin.to_crs(crs='EPSG:25833', inplace=True)
x, y = round(subbasin.geometry.x[0], 1), round(subbasin.geometry.y[0], 1)
# subbasin.to_file('subbasin.geojson', driver='GeoJSON')
folder = os.path.normpath(os.path.join(PROJECT_DES, test_folder))
if not os.path.exists(folder): os.makedirs(folder)
cwd_path = os.path.normpath(os.path.join(folder, 'wflow_model'))
if not os.path.exists(cwd_path): os.makedirs(cwd_path)
# Copy config file
shutil.copy(r"backend\src\flow_samples\config.yml", folder)
wflow_exe = os.path.normpath(os.path.join(WFLOW_PATH, "wflow_cli", "bin", "wflow_cli.exe"))
# region = {'subbasin': [55010.153, 6955129.102]}   'subbasin': [55010.0, 6955130.0]
region = {'subbasin': [x, y]}
soil_layers, time_zone = [50, 100, 150, 300, 400, 600], 'Europe/Oslo'
data_lib, uparea = [os.path.normpath(os.path.join(folder, 'config.yml'))], 10
params = {
    "k_sat_ratio": 100,          # subsurface horizontal/vertical conductivity ratio
    "dd_snow": 3.75653,          # snow degree-day coefficient
    "inf_red": 0.038,            # infiltration reduction parameter
    "canopy_evap_ratio": 0.11,   # canopy evaporation / precipitation ratio
    "inf_cap": 10,               # compacted soil infiltration capacity
    "leak_max": 0,               # max leakage flux bottom soil
    "soil_sigmoid": -500,        # root zone sigmoid shape parameter
    "snowfall_t0": 0,            # snowfall temp threshold
    "snowfall_dt": 2,            # snowfall temp interval
    "melt_t0_snow": 0,           # snow melt threshold temp
    "snow_liq_cap": 0.1,         # snow liquid water holding capacity
    "dd_glacier": 3,             # glacier degree-day coefficient
    "firn_dry_frac": 0.001,      # firn dry snow depth fraction
    "melt_t0_glacier": 0         # glacier melt threshold temp
}
run_hydromt(
    cwd_path, start, end, time_zone, step, region, 10, soil_layers, data_lib, uparea, params,
)
# Run simulation
cmd = [wflow_exe, os.path.normpath(os.path.join(cwd_path, 'wflow_sbm.toml'))]
process = subprocess.Popen(
    cmd, cwd=cwd_path, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout: print(line, end='')
process.wait()
print('Return code:', process.returncode)